# 🏠 House Price Prediction — Linear Regression
**Dataset:** California Housing (built into scikit-learn)  
**Algorithm:** Linear Regression  
**Goal:** Predict median house value based on features like income, location, and house age.

> **Why Linear Regression?** The target variable (price) is continuous, and we expect a roughly linear relationship between income and house prices — making this a perfect use case.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load dataset
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

print("Shape:", df.shape)
df.head()


## Step 1 — Understand the Data

In [ ]:
print(df.describe())
print("\nMissing values:", df.isnull().sum().sum())


## Step 2 — Visualize Target Distribution

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df['MedHouseVal'], bins=50, color='#3498DB', edgecolor='black', alpha=0.7)
plt.title('Distribution of Median House Values', fontsize=14)
plt.xlabel('Median House Value ($100,000s)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


## Step 3 — Feature Correlation with Target

In [ ]:
corr = df.corr()['MedHouseVal'].drop('MedHouseVal').sort_values()

plt.figure(figsize=(8, 5))
corr.plot(kind='barh', color=['#E74C3C' if c < 0 else '#2ECC71' for c in corr])
plt.title('Feature Correlation with House Value', fontsize=13)
plt.xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

print(corr)


**Finding:** `MedInc` (median income) has the highest positive correlation with house value — makes intuitive sense!


## Step 4 — Preprocessing & Train/Test Split

In [ ]:
X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (important for linear models)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # use same scaler, don't refit on test!

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")


**Why scale?** Linear regression is sensitive to feature magnitude. Income values in tens of thousands and latitude values around 37 are on very different scales — scaling ensures no feature dominates unfairly.


## Step 5 — Train the Model

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

print("Model trained successfully!")
print(f"\nFeature Coefficients:")
for feat, coef in zip(X.columns, model.coef_):
    print(f"  {feat:15s}: {coef:+.4f}")
print(f"  Intercept      : {model.intercept_:.4f}")


## Step 6 — Evaluate the Model

In [ ]:
y_pred = model.predict(X_test_scaled)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("=" * 40)
print("       MODEL EVALUATION RESULTS")
print("=" * 40)
print(f"  R² Score  : {r2:.4f}  (1.0 = perfect)")
print(f"  RMSE      : {rmse:.4f}")
print(f"  MAE       : {mae:.4f}")
print("=" * 40)


**Interpreting Results:**
- **R²** tells us what % of variance the model explains. ~0.60 means the model explains 60% of price variation.
- **RMSE** is in the same units as house price ($100k). Lower = better.
- **MAE** is average absolute prediction error.


## Step 7 — Actual vs Predicted Plot

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='#3498DB', edgecolors='none')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Prices', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()


## Summary
- Linear Regression gives a reasonable baseline for house price prediction
- The model struggles with high-value houses (right side of plot) — they're harder to predict linearly
- **Next step:** Try Ridge/Lasso Regression or Random Forest for better performance
